In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

STUDENT_ID = "TUPM-26-1129"  
LAST_NAME = "Espejo"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(digit) for digit in id_digits)
NAME_LENGTH = len(LAST_NAME)

# Keep this formula consistent with the main laboratory activity.
RANDOM_SEED = ID_SUM + SEED_DIGIT + NAME_LENGTH

print("STUDENT CONFIGURATION")
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)


STUDENT CONFIGURATION
Seed Digit: 9
ID Sum: 21
Name Length: 6
Random Seed: 36


In [2]:
CURRENT_DIR = Path.cwd()
PROJECT_DIR = None

for folder in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if ((folder / "data" / "secom.data").exists()
        and (folder / "data" / "secom_labels.data").exists()):
        PROJECT_DIR = folder
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Place secom.data and secom_labels.data inside the main project's data folder."
    )

DATA_PATH = PROJECT_DIR / "data" / "secom.data"
LABEL_PATH = PROJECT_DIR / "data" / "secom_labels.data"
OUTPUT_DIR = PROJECT_DIR / "Exercise_2_Mobile_Device_Profiling" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

measurements = pd.read_csv(DATA_PATH, sep=r"\s+", header=None)
measurements.columns = [f"Feature_{i+1}" for i in range(measurements.shape[1])]

labels_raw = pd.read_csv(LABEL_PATH, sep=r"\s+", header=None)
raw_labels = pd.to_numeric(labels_raw.iloc[:, 0], errors="coerce")

if raw_labels.isna().any():
    raise ValueError("The first column of secom_labels.data contains invalid labels.")

# SECOM label convention: -1 = Pass, 1 = Fail
outcomes = raw_labels.map({-1: "Pass", 1: "Fail"})
if outcomes.isna().any():
    raise ValueError("Expected label values -1 (Pass) and 1 (Fail).")

if len(measurements) != len(outcomes):
    raise ValueError("Measurement and label files have different row counts.")

data = measurements.copy()
data.insert(0, "Record_ID", np.arange(1, len(data) + 1))
data["Outcome"] = outcomes.to_numpy()
feature_columns = [c for c in data.columns if c.startswith("Feature_")]

print("Records:", len(data))
print("Measurement features:", len(feature_columns))
print("Pass records:", (data["Outcome"] == "Pass").sum())
print("Fail records:", (data["Outcome"] == "Fail").sum())
data.head()


Records: 1567
Measurement features: 590
Pass records: 1463
Fail records: 104


,Record_ID,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,...,Feature_582,Feature_583,Feature_584,Feature_585,Feature_586,Feature_587,Feature_588,Feature_589,Feature_590,Outcome
0,1,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,Pass
1,2,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,Pass
2,3,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,Fail
3,4,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,Pass
4,5,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,Pass


In [3]:
X_full = data[feature_columns].apply(pd.to_numeric, errors="coerce")

usable_features = [
    f for f in feature_columns
    if X_full[f].notna().any() and X_full[f].nunique(dropna=True) > 1
]

if len(usable_features) < 5:
    raise ValueError("Fewer than five usable measurement features were found.")

rng = np.random.default_rng(RANDOM_SEED)
selected_features = list(rng.choice(usable_features, size=5, replace=False))

print("Number of usable features:", len(usable_features))
print("Selected features:")
for f in selected_features:
    print("-", f)


Number of usable features: 474
Selected features:
- Feature_199
- Feature_219
- Feature_92
- Feature_207
- Feature_226


In [4]:
pass_records = data[data["Outcome"] == "Pass"]
fail_records = data[data["Outcome"] == "Fail"]

# Select this many records from EACH outcome group
sample_per_group = min(20 + ID_SUM, len(pass_records), len(fail_records))
if sample_per_group == 0:
    raise ValueError("Both Pass and Fail groups must contain records.")

pass_sample = pass_records.sample(n=sample_per_group, random_state=RANDOM_SEED)
fail_sample = fail_records.sample(n=sample_per_group, random_state=RANDOM_SEED + 1)

balanced_sample = pd.concat([pass_sample, fail_sample], axis=0)
balanced_sample = balanced_sample.sample(
    frac=1, random_state=RANDOM_SEED + 2
).reset_index(drop=True)

print("Pass records selected:", (balanced_sample["Outcome"] == "Pass").sum())
print("Fail records selected:", (balanced_sample["Outcome"] == "Fail").sum())
print("Total balanced sample size:", len(balanced_sample))
balanced_sample[["Record_ID", "Outcome"]].head(10)

Pass records selected: 41
Fail records selected: 41
Total balanced sample size: 82


,Record_ID,Outcome
0,278,Fail
1,328,Fail
2,54,Pass
3,906,Pass
4,1032,Pass
5,357,Pass
6,1436,Pass
7,51,Fail
8,39,Fail
9,1150,Pass


In [5]:
# Calculate medians from the FULL dataset, not the balanced sample
full_dataset_medians = X_full[selected_features].median()

balanced_analysis = balanced_sample.copy()
balanced_analysis[selected_features] = (
    balanced_analysis[selected_features]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(full_dataset_medians)
)

print("Full-dataset medians used for imputation:")
display(full_dataset_medians.to_frame(name="Full_Dataset_Median"))
print("Missing values remaining in selected sample features:",
      int(balanced_analysis[selected_features].isna().sum().sum()))

balanced_analysis[["Record_ID", "Outcome"] + selected_features].head()


Full-dataset medians used for imputation:


,Full_Dataset_Median
Feature_199,0.42400
Feature_219,3.63075
Feature_92,0.00000
Feature_207,0.00000
Feature_226,967.29980


Missing values remaining in selected sample features: 0


,Record_ID,Outcome,Feature_199,Feature_219,Feature_92,Feature_207,Feature_226
0,278,Fail,0.878,5.4749,-0.0139,0.0,1351.3994
1,328,Fail,0.260,3.3942,0.0515,0.0,679.0000
2,54,Pass,0.324,3.7162,-0.0126,0.0,776.3994
3,906,Pass,0.446,1.8895,-0.0185,0.0,691.5996
4,1032,Pass,0.284,2.6169,0.1479,0.0,842.4004


In [6]:
grouped_statistics = (
    balanced_analysis
    .groupby("Outcome")[selected_features]
    .agg(["count", "mean", "median", "std"])
)

display(grouped_statistics)


Feature_199                            Feature_219                    \
              count      mean median       std       count      mean  median   
Outcome                                                                        
Fail             41  0.812610  0.503  1.370495          41  3.634676  3.6033   
Pass             41  0.502073  0.380  0.288963          41  3.611288  3.8260   

                  Feature_92                             Feature_207       \
              std      count      mean  median       std       count mean   
Outcome                                                                     
Fail     1.298211         41 -0.000849  0.0034  0.078738          41  0.0   
Pass     1.033460         41 -0.009402 -0.0074  0.088386          41  0.0   

                    Feature_226                                     
        median  std       count         mean    median         std  
Outcome                                                             
Fail       0.0  0.0          41  1112.158524  965.9004  553.484099  
Pass       0.0  0.0          41  1103.902415  967.9004  419.605627

In [7]:
comparison_rows = []

for feature in selected_features:
    pass_values = balanced_analysis.loc[
        balanced_analysis["Outcome"] == "Pass", feature
    ].to_numpy(dtype=float)
    fail_values = balanced_analysis.loc[
        balanced_analysis["Outcome"] == "Fail", feature
    ].to_numpy(dtype=float)

    n_pass, n_fail = len(pass_values), len(fail_values)
    pass_mean, fail_mean = np.mean(pass_values), np.mean(fail_values)
    pass_var = np.var(pass_values, ddof=1)
    fail_var = np.var(fail_values, ddof=1)

    # Pooled within-group sample variance and standard deviation
    pooled_variance = (
        ((n_pass - 1) * pass_var + (n_fail - 1) * fail_var)
        / (n_pass + n_fail - 2)
    )
    pooled_std = np.sqrt(pooled_variance)

    # Difference is Fail mean minus Pass mean
    mean_difference = fail_mean - pass_mean

    # Safely handle a zero pooled standard deviation
    if np.isclose(pooled_std, 0):
        standardized_difference = (
            0.0 if np.isclose(mean_difference, 0) else np.nan
        )
    else:
        standardized_difference = mean_difference / pooled_std

    comparison_rows.append({
        "Feature": feature,
        "Pass_Count": n_pass,
        "Pass_Mean": pass_mean,
        "Pass_Median": np.median(pass_values),
        "Pass_Sample_Std": np.std(pass_values, ddof=1),
        "Fail_Count": n_fail,
        "Fail_Mean": fail_mean,
        "Fail_Median": np.median(fail_values),
        "Fail_Sample_Std": np.std(fail_values, ddof=1),
        "Mean_Difference_Fail_Minus_Pass": mean_difference,
        "Absolute_Mean_Difference": abs(mean_difference),
        "Standardized_Mean_Difference": standardized_difference,
        "Absolute_Standardized_Mean_Difference": (
            abs(standardized_difference)
            if np.isfinite(standardized_difference) else np.nan
        )
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)


,Feature,Pass_Count,Pass_Mean,Pass_Median,Pass_Sample_Std,Fail_Count,Fail_Mean,Fail_Median,Fail_Sample_Std,Mean_Difference_Fail_Minus_Pass,Absolute_Mean_Difference,Standardized_Mean_Difference,Absolute_Standardized_Mean_Difference
0,Feature_199,41,0.502073,0.3800,0.288963,41,0.812610,0.5030,1.370495,0.310537,0.310537,0.313549,0.313549
1,Feature_219,41,3.611288,3.8260,1.033460,41,3.634676,3.6033,1.298211,0.023388,0.023388,0.019933,0.019933
2,Feature_92,41,-0.009402,-0.0074,0.088386,41,-0.000849,0.0034,0.078738,0.008554,0.008554,0.102193,0.102193
3,Feature_207,41,0.000000,0.0000,0.000000,41,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Feature_226,41,1103.902415,967.9004,419.605627,41,1112.158524,965.9004,553.484099,8.256110,8.256110,0.016811,0.016811


In [8]:
checks_by_feature = []

for feature in selected_features:
    pass_values = balanced_analysis.loc[
        balanced_analysis["Outcome"] == "Pass", feature
    ].to_numpy(dtype=float)
    fail_values = balanced_analysis.loc[
        balanced_analysis["Outcome"] == "Fail", feature
    ].to_numpy(dtype=float)

    p_pass = grouped_statistics.loc["Pass", feature]
    p_fail = grouped_statistics.loc["Fail", feature]

    checks = [
        len(pass_values) == int(p_pass["count"]),
        len(fail_values) == int(p_fail["count"]),
        np.isclose(np.mean(pass_values), p_pass["mean"]),
        np.isclose(np.mean(fail_values), p_fail["mean"]),
        np.isclose(np.median(pass_values), p_pass["median"]),
        np.isclose(np.median(fail_values), p_fail["median"]),
        np.isclose(np.std(pass_values, ddof=1), p_pass["std"]),
        np.isclose(np.std(fail_values, ddof=1), p_fail["std"])
    ]
    checks_by_feature.append(all(checks))

comparison["NumPy_Pandas_Statistics_Match"] = checks_by_feature

comparison = comparison.sort_values(
    "Absolute_Standardized_Mean_Difference",
    ascending=False,
    na_position="last"
).reset_index(drop=True)
comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

statistics_match = bool(comparison["NumPy_Pandas_Statistics_Match"].all())
display(comparison)
print("NumPy-Pandas Statistics Match:", statistics_match)


,Rank,Feature,Pass_Count,Pass_Mean,Pass_Median,Pass_Sample_Std,Fail_Count,Fail_Mean,Fail_Median,Fail_Sample_Std,Mean_Difference_Fail_Minus_Pass,Absolute_Mean_Difference,Standardized_Mean_Difference,Absolute_Standardized_Mean_Difference,NumPy_Pandas_Statistics_Match
0,1,Feature_199,41,0.502073,0.3800,0.288963,41,0.812610,0.5030,1.370495,0.310537,0.310537,0.313549,0.313549,True
1,2,Feature_92,41,-0.009402,-0.0074,0.088386,41,-0.000849,0.0034,0.078738,0.008554,0.008554,0.102193,0.102193,True
2,3,Feature_219,41,3.611288,3.8260,1.033460,41,3.634676,3.6033,1.298211,0.023388,0.023388,0.019933,0.019933,True
3,4,Feature_226,41,1103.902415,967.9004,419.605627,41,1112.158524,965.9004,553.484099,8.256110,8.256110,0.016811,0.016811,True
4,5,Feature_207,41,0.000000,0.0000,0.000000,41,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,True


NumPy-Pandas Statistics Match: True


In [9]:
balanced_output = OUTPUT_DIR / "balanced_sample.csv"
comparison_output = OUTPUT_DIR / "feature_comparison.csv"

# Preserve original record IDs and outcome labels in the balanced sample file
balanced_sample[["Record_ID", "Outcome"] + feature_columns].to_csv(
    balanced_output, index=False
)
comparison.to_csv(comparison_output, index=False)

print("Balanced sample saved to:", balanced_output)
print("Feature comparison saved to:", comparison_output)


Balanced sample saved to: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_2_Mobile_Device_Profiling\output\balanced_sample.csv
Feature comparison saved to: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_2_Mobile_Device_Profiling\output\feature_comparison.csv


In [10]:
highest_ranked_feature = comparison.iloc[0]["Feature"]
highest_absolute_smd = comparison.iloc[0][
    "Absolute_Standardized_Mean_Difference"
]

assessment = pd.DataFrame({
    "Output Parameter": [
        "Balanced Sample Size",
        "Selected Features",
        "Highest-Ranked Feature",
        "Absolute Standardized Mean Difference",
        "NumPy-Pandas Statistics Match"
    ],
    "Observed Value": [
        len(balanced_sample),
        ", ".join(selected_features),
        highest_ranked_feature,
        (
            f"{highest_absolute_smd:.6f}"
            if pd.notna(highest_absolute_smd)
            else "Undefined (zero pooled standard deviation)"
        ),
        statistics_match
    ]
})

print("=" * 70)
print("EXERCISE 2: ASSESSMENT DATA")
print("=" * 70)
display(assessment)

print("REQUIRED OUTPUTS")
print("1. balanced_sample.csv")
print("2. feature_comparison.csv")
print("Output folder:", OUTPUT_DIR)
print("=" * 70)


EXERCISE 2: ASSESSMENT DATA


,Output Parameter,Observed Value
0,Balanced Sample Size,82
1,Selected Features,"Feature_199, Feature_219, Feature_92, Feature_..."
2,Highest-Ranked Feature,Feature_199
3,Absolute Standardized Mean Difference,0.313549
4,NumPy-Pandas Statistics Match,True


REQUIRED OUTPUTS
1. balanced_sample.csv
2. feature_comparison.csv
Output folder: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_2_Mobile_Device_Profiling\output
